asset feature
종목별 정보 9개
        +
market feature
krx300 시장 상태
        +
macro feature
금리 / 환율 / 국고채
        ->
date × ticker common dataset

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\code\portfolio_optimization")

print(PROJECT_ROOT)

C:\code\portfolio_optimization


In [40]:
# 05d-1. 데이터 경로 설정

ASSET_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "assets"
    / "weekly_asset_features.parquet"
)

MARKET_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "krx300_market_features.csv"
)

MACRO_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "macro_features.csv"
)

KRX_STOCK_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "stocks"
    / "krx_stock_panel_clean.parquet"
)

print(
    "stock panel exists:",
    KRX_STOCK_PANEL_PATH.exists()
)

stock panel exists: True


In [6]:
# 05d-2. asset feature 불러오기

import pyarrow.parquet as pq

asset_features = (
    pq.read_table(
        ASSET_FEATURE_PATH
    )
    .to_pandas()
)


print(
    asset_features.shape
)

print(
    asset_features[
        "signal_date"
    ].min()
)

print(
    asset_features[
        "signal_date"
    ].max()
)

(21900, 15)
2018-05-04 00:00:00
2026-09-15 00:00:00


In [7]:
# 05d-3. market feature 불러오기

market_features = pd.read_csv(
    MARKET_FEATURE_PATH
)

market_features["date"] = pd.to_datetime(
    market_features["date"]
)


print(
    market_features.shape
)

print(
    market_features["date"].min()
)

print(
    market_features["date"].max()
)

(2112, 17)
2018-02-05 00:00:00
2026-09-15 00:00:00


In [8]:
# 05d-4. macro feature 불러오기

macro_features = pd.read_csv(
    MACRO_FEATURE_PATH
)

macro_features["date"] = pd.to_datetime(
    macro_features["date"]
)


print(
    macro_features.shape
)

print(
    macro_features["date"].min()
)

print(
    macro_features["date"].max()
)

(2112, 12)
2018-02-05 00:00:00
2026-09-15 00:00:00


In [20]:
# 05d-5. asset feature 목록 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

In [21]:
# 05d-6. market feature 목록 설정

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "trading_value_ratio_20d"
]

In [22]:
# 05d-7. macro feature 목록 설정

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]

In [23]:
# 05d-8. macro feature 연결

common_dataset = (
    common_dataset
    .merge(
        macro_features[
            [
                "date"
            ]
            + MACRO_FEATURES
        ],
        left_on="signal_date",
        right_on="date",
        how="left"
    )
    .drop(
        columns=[
            "date"
        ]
    )
)

In [24]:
# 05d-9. common dataset 기본 검증

print(
    "shape:",
    common_dataset.shape
)

print(
    "signals:",
    common_dataset[
        "signal_date"
    ].nunique()
)

print(
    "tickers:",
    common_dataset[
        "ticker"
    ].nunique()
)

print(
    "duplicates:",
    common_dataset[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)


shape: (21900, 45)
signals: 438
tickers: 309
duplicates: 0


In [26]:
# 05d-10a. 현재 common dataset 컬럼 확인

print(
    common_dataset.columns.tolist()
)

['signal_date', 'execution_date', 'ticker', 'name', 'market', 'liquidity_rank', 'return_1d', 'return_5d', 'momentum_20d', 'momentum_60d', 'volatility_20d', 'drawdown_20d', 'trading_value_ma20', 'trading_value_ratio_20d_x', 'log_market_cap', 'market_return_1d', 'market_return_5d', 'market_return_20d', 'market_volatility_20d', 'market_drawdown', 'volume_change_1d', 'trading_value_change_1d', 'trading_value_ratio_20d_y', 'base_rate_x', 'usdkrw_x', 'bond3y_x', 'usdkrw_return_1d_x', 'usdkrw_return_5d_x', 'usdkrw_return_20d_x', 'bond3y_change_1d_x', 'bond3y_change_5d_x', 'bond3y_change_20d_x', 'base_rate_change_x', 'rate_spread_3y_x', 'base_rate_y', 'usdkrw_y', 'bond3y_y', 'usdkrw_return_1d_y', 'usdkrw_return_5d_y', 'usdkrw_return_20d_y', 'bond3y_change_1d_y', 'bond3y_change_5d_y', 'bond3y_change_20d_y', 'base_rate_change_y', 'rate_spread_3y_y']


In [27]:
# trading value 관련 컬럼 확인

print(
    [
        col
        for col in common_dataset.columns
        if "trading_value" in col
    ]
)

['trading_value_ma20', 'trading_value_ratio_20d_x', 'trading_value_change_1d', 'trading_value_ratio_20d_y']


asset trading_value_ratio_20d = 개별 종목 거래대금 비율
market trading_value_ratio_20d = KRX300 전체 시장 거래대금 비율

In [28]:
# 05d-7. asset dataset 기준으로 common dataset 생성

common_dataset = (
    asset_features
    .copy()
)

print(
    "shape:",
    common_dataset.shape
)

shape: (21900, 15)


In [29]:
# 05d-8. market feature 이름 정리

market_features_for_merge = (
    market_features[
        [
            "date",
            "market_return_1d",
            "market_return_5d",
            "market_return_20d",
            "market_volatility_20d",
            "market_drawdown",
            "volume_change_1d",
            "trading_value_change_1d",
            "trading_value_ratio_20d"
        ]
    ]
    .rename(
        columns={
            "trading_value_ratio_20d":
            "market_trading_value_ratio_20d"
        }
    )
)

In [30]:
# market feature 목록 설정

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

In [31]:
# 05d-9. market feature 연결

common_dataset = (
    common_dataset
    .merge(
        market_features_for_merge,
        left_on="signal_date",
        right_on="date",
        how="left"
    )
    .drop(
        columns=[
            "date"
        ]
    )
)

print(
    "after market:",
    common_dataset.shape
)

after market: (21900, 23)


In [32]:
# 05d-10. macro 컬럼 확인

print(
    macro_features.columns.tolist()
)

['date', 'base_rate', 'usdkrw', 'bond3y', 'usdkrw_return_1d', 'bond3y_change_1d', 'base_rate_change', 'usdkrw_return_5d', 'usdkrw_return_20d', 'bond3y_change_5d', 'bond3y_change_20d', 'rate_spread_3y']


In [33]:
# macro feature 목록 설정

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]

In [34]:
# 05d-11. macro feature 연결

common_dataset = (
    common_dataset
    .merge(
        macro_features[
            [
                "date"
            ]
            + MACRO_FEATURES
        ],
        left_on="signal_date",
        right_on="date",
        how="left"
    )
    .drop(
        columns=[
            "date"
        ]
    )
)


print(
    "after macro:",
    common_dataset.shape
)

after macro: (21900, 34)


In [35]:
# 05d-12. model feature 컬럼 존재 확인

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]


ALL_MODEL_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)


missing_columns = [
    col
    for col in ALL_MODEL_FEATURES
    if col not in common_dataset.columns
]


print(
    "missing columns:",
    missing_columns
)

missing columns: []


In [36]:
# 05d-13. common feature 결측 확인

missing_summary = (
    common_dataset[
        ALL_MODEL_FEATURES
    ]
    .isna()
    .sum()
)


print(
    missing_summary[
        missing_summary > 0
    ]
)

Series([], dtype: int64)


asset
trading_value_ratio_20d

market
market_trading_value_ratio_20d

In [37]:
# 05d-14. common feature inf 확인

inf_summary = pd.Series({
    col: np.isinf(
        common_dataset[col]
    ).sum()
    for col in ALL_MODEL_FEATURES
})

print(
    inf_summary[
        inf_summary > 0
    ]
)

Series([], dtype: int64)


In [38]:
# 05d-15. rebalance schedule 생성

rebalance_schedule = (
    common_dataset[
        [
            "signal_date",
            "execution_date"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "signal_date"
    )
    .reset_index(
        drop=True
    )
)

rebalance_schedule["next_signal_date"] = (
    rebalance_schedule[
        "signal_date"
    ]
    .shift(-1)
)

rebalance_schedule["next_execution_date"] = (
    rebalance_schedule[
        "execution_date"
    ]
    .shift(-1)
)

print(
    "schedules:",
    len(rebalance_schedule)
)

rebalance_schedule.head()

schedules: 438


,signal_date,execution_date,next_signal_date,next_execution_date
0,2018-05-04,2018-05-08,2018-05-11,2018-05-14
1,2018-05-11,2018-05-14,2018-05-18,2018-05-21
2,2018-05-18,2018-05-21,2018-05-25,2018-05-28
3,2018-05-25,2018-05-28,2018-06-01,2018-06-04
4,2018-06-01,2018-06-04,2018-06-08,2018-06-11


In [41]:
# 05d-16. target 계산용 가격 데이터 불러오기

target_tickers = (
    common_dataset[
        "ticker"
    ]
    .drop_duplicates()
    .tolist()
)


target_price_table = pq.read_table(
    KRX_STOCK_PANEL_PATH,
    columns=[
        "date",
        "ticker",
        "open",
        "close",
        "change_pct"
    ],
    filters=[
        (
            "ticker",
            "in",
            target_tickers
        )
    ]
)


target_price_history = (
    target_price_table
    .to_pandas()
    .sort_values(
        [
            "ticker",
            "date"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "rows:",
    len(target_price_history)
)

print(
    "tickers:",
    target_price_history[
        "ticker"
    ].nunique()
)

rows: 605898
tickers: 309


In [43]:
# 05d-17. krx return index 생성
# return_index: KRX 일간 수익률을 계속 복리 누적한 가상의 가격지수

target_price_history["return_1d"] = (
    target_price_history[
        "change_pct"
    ]
    / 100
)


target_price_history["return_index"] = (
    target_price_history
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .transform(
        lambda x: (
            1 + x
        ).cumprod()
    )
)

In [45]:
# 05d-18. adjustment factor 생성
# adjustment factor = 조정계수

target_price_history["adjustment_factor"] = ( 
    target_price_history[
        "return_index"
    ]
    / target_price_history[
        "close"
    ]
)

In [46]:
# 05d-19. adjusted open 생성

target_price_history["adjusted_open"] = np.where(
    target_price_history["open"] > 0,
    target_price_history["open"]
    * target_price_history[
        "adjustment_factor"
    ],
    np.nan
)

In [47]:
# 05d-20. 다음 execution date 연결

common_dataset_target = (
    common_dataset
    .merge(
        rebalance_schedule[
            [
                "signal_date",
                "execution_date",
                "next_signal_date",
                "next_execution_date"
            ]
        ],
        on=[
            "signal_date",
            "execution_date"
        ],
        how="left"
    )
)


print(
    common_dataset_target.shape
)

(21900, 36)


In [48]:
# 05d-21. 현재 execution open 연결

execution_prices = (
    target_price_history[
        [
            "date",
            "ticker",
            "adjusted_open"
        ]
    ]
    .rename(
        columns={
            "date": "execution_date",
            "adjusted_open": "execution_open"
        }
    )
)


common_dataset_target = (
    common_dataset_target
    .merge(
        execution_prices,
        on=[
            "execution_date",
            "ticker"
        ],
        how="left"
    )
)

In [49]:
# 05d-22. 다음 execution open 연결

next_execution_prices = (
    target_price_history[
        [
            "date",
            "ticker",
            "adjusted_open"
        ]
    ]
    .rename(
        columns={
            "date": "next_execution_date",
            "adjusted_open": "next_execution_open"
        }
    )
)


common_dataset_target = (
    common_dataset_target
    .merge(
        next_execution_prices,
        on=[
            "next_execution_date",
            "ticker"
        ],
        how="left"
    )
)

In [50]:
# 05d-23. next rebalance target return 계산

common_dataset_target["target_return"] = (
    common_dataset_target[
        "next_execution_open"
    ]
    / common_dataset_target[
        "execution_open"
    ]
    - 1
)

In [51]:
# 05d-24. target 결측 확인

target_missing = (
    common_dataset_target[
        common_dataset_target[
            "target_return"
        ]
        .isna()
    ]
    .copy()
)


print(
    "missing target:",
    len(target_missing)
)

print(
    "\nmissing by signal:"
)

print(
    target_missing[
        "signal_date"
    ]
    .value_counts()
    .sort_index()
    .tail(20)
)

missing target: 121

missing by signal:
signal_date
2019-01-04     1
2020-04-29     1
2020-12-11     1
2020-12-18     1
2021-04-02     1
2021-04-09     1
2021-06-25     1
2021-10-22     1
2023-02-17     1
2023-02-24     1
2023-12-08     1
2023-12-15     1
2024-04-05     1
2024-08-23     1
2025-03-14     1
2025-03-21     1
2025-10-24     1
2025-11-21     1
2026-09-11    50
2026-09-15    50
Name: count, dtype: int64


In [52]:
# 05d-25. target 결측 원인 확인

print(
    target_missing[
        [
            "signal_date",
            "execution_date",
            "next_execution_date",
            "ticker",
            "execution_open",
            "next_execution_open"
        ]
    ]
    .head(60)
)

      signal_date execution_date next_execution_date  ticker  execution_open  \
1072   2018-09-28     2018-10-01          2018-10-08  035420        0.792923   
1123   2018-10-05     2018-10-08          2018-10-15  035420             NaN   
1356   2018-11-09     2018-11-12          2018-11-19  207940        0.775050   
1795   2019-01-04     2019-01-07          2019-01-14  000030        0.922750   
5228   2020-04-29     2020-05-04          2020-05-11  215600        0.117006   
6817   2020-12-11     2020-12-14          2020-12-21  003090        2.930423   
6859   2020-12-18     2020-12-21          2020-12-28  003090             NaN   
7609   2021-04-02     2021-04-05          2021-04-12  035720        3.505271   
7656   2021-04-09     2021-04-12          2021-04-19  035720             NaN   
8214   2021-06-25     2021-06-28          2021-07-05  042670        1.486002   
9073   2021-10-22     2021-10-25          2021-11-01  017670        1.204970   
12524  2023-02-17     2023-02-20        

In [53]:
# 05d-26. target 사용 가능한 dataset 생성

model_dataset = (
    common_dataset_target[
        common_dataset_target[
            "target_return"
        ]
        .notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    model_dataset.shape
)

print(
    "signals:",
    model_dataset[
        "signal_date"
    ].nunique()
)

print(
    "rows per signal:"
)

print(
    model_dataset
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .nunique()
    .describe()
)

shape: (21779, 39)
signals: 436
rows per signal:
count    436.000000
mean      49.951835
std        0.214361
min       49.000000
25%       50.000000
50%       50.000000
75%       50.000000
max       50.000000
Name: ticker, dtype: float64


common_dataset
438 signals × 50 stocks
feature/state 데이터
↓
항상 50종목 유지
─────────────────────────
supervised_dataset
target_return 존재하는 행만
Ridge / RF / XGBoost 학습용
─────────────────────────
backtest / RL dataset
50종목 유지
execution 불가 상황은
실제 운용 규칙으로 처리

In [54]:
# 05d-26. target 결측 원인 분류

target_missing = (
    common_dataset_target[
        common_dataset_target["target_return"].isna()
    ]
    .copy()
)


target_missing["missing_reason"] = np.select(
    [
        target_missing["next_execution_date"].isna(),
        (
            target_missing["execution_open"].isna()
            &
            target_missing["next_execution_open"].notna()
        ),
        (
            target_missing["execution_open"].notna()
            &
            target_missing["next_execution_open"].isna()
        ),
        (
            target_missing["execution_open"].isna()
            &
            target_missing["next_execution_open"].isna()
        )
    ],
    [
        "no_future_horizon",
        "missing_entry_open",
        "missing_exit_open",
        "missing_both_open"
    ],
    default="other"
)


print(
    target_missing[
        "missing_reason"
    ]
    .value_counts()
)

missing_reason
no_future_horizon     100
missing_exit_open      15
missing_entry_open      3
missing_both_open       3
Name: count, dtype: int64


In [55]:
# 05d-27. 실제 execution 문제만 확인

execution_missing = (
    target_missing[
        target_missing["missing_reason"]
        != "no_future_horizon"
    ]
    .copy()
)


print(
    "execution missing:",
    len(execution_missing)
)


print(
    execution_missing[
        [
            "signal_date",
            "execution_date",
            "next_execution_date",
            "ticker",
            "name",
            "execution_open",
            "next_execution_open",
            "missing_reason"
        ]
    ]
    .to_string(
        index=False
    )
)

execution missing: 21
signal_date execution_date next_execution_date ticker      name  execution_open  next_execution_open     missing_reason
 2018-09-28     2018-10-01          2018-10-08 035420     NAVER        0.792923                  NaN  missing_exit_open
 2018-10-05     2018-10-08          2018-10-15 035420     NAVER             NaN             0.763572 missing_entry_open
 2018-11-09     2018-11-12          2018-11-19 207940  삼성바이오로직스        0.775050                  NaN  missing_exit_open
 2019-01-04     2019-01-07          2019-01-14 000030      우리은행        0.922750                  NaN  missing_exit_open
 2020-04-29     2020-05-04          2020-05-11 215600       신라젠        0.117006                  NaN  missing_exit_open
 2020-12-11     2020-12-14          2020-12-21 003090        대웅        2.930423                  NaN  missing_exit_open
 2020-12-18     2020-12-21          2020-12-28 003090        대웅             NaN             2.563605 missing_entry_open
 2021-04-02     20

In [56]:
# 05d-28. supervised learning dataset 생성
# 미래 horizon이 존재하고 target 계산 가능한 행만 사용

supervised_dataset = (
    common_dataset_target[
        common_dataset_target[
            "target_return"
        ]
        .notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    supervised_dataset.shape
)

print(
    "signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

print(
    "target missing:",
    supervised_dataset[
        "target_return"
    ].isna()
    .sum()
)

shape: (21779, 39)
signals: 436
target missing: 0


일부 signal은 49개 training samples만 존재
= universe가 49개라는 뜻 아님
= 50개 중 1개 종목의 forward target을 관측할 수 없었다는 뜻

In [57]:
# 05d-29. model state dataset 생성
# 실제 universe 구조는 50종목 그대로 유지

state_dataset = (
    common_dataset
    .copy()
)


state_count = (
    state_dataset
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .nunique()
)


print(
    "shape:",
    state_dataset.shape
)

print(
    "signals:",
    state_dataset[
        "signal_date"
    ].nunique()
)

print(
    state_count.describe()
)

shape: (21900, 34)
signals: 438
count    438.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
Name: ticker, dtype: float64


In [58]:
# 05d-30. 저장 경로 설정

COMMON_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
)

COMMON_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STATE_DATASET_PATH = (
    COMMON_FEATURE_DIR
    / "state_dataset.parquet"
)

SUPERVISED_DATASET_PATH = (
    COMMON_FEATURE_DIR
    / "supervised_dataset.parquet"
)

TARGET_AUDIT_PATH = (
    COMMON_FEATURE_DIR
    / "target_missing_audit.csv"
)

In [60]:
# 05d-31. state dataset 저장

import pyarrow as pa

state_table = pa.Table.from_pandas(
    state_dataset,
    preserve_index=False
)

pq.write_table(
    state_table,
    STATE_DATASET_PATH,
    compression="snappy"
)

print(
    "saved:",
    STATE_DATASET_PATH
)

saved: C:\code\portfolio_optimization\data\features\common\state_dataset.parquet


In [61]:
# 05d-32. supervised dataset 저장

supervised_table = pa.Table.from_pandas(
    supervised_dataset,
    preserve_index=False
)

pq.write_table(
    supervised_table,
    SUPERVISED_DATASET_PATH,
    compression="snappy"
)

print(
    "saved:",
    SUPERVISED_DATASET_PATH
)

saved: C:\code\portfolio_optimization\data\features\common\supervised_dataset.parquet


In [62]:
# 05d-33. target 결측 audit 저장

target_missing.to_csv(
    TARGET_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "saved:",
    TARGET_AUDIT_PATH
)

saved: C:\code\portfolio_optimization\data\features\common\target_missing_audit.csv


In [63]:
# 05d-34. 저장 결과 검증

state_check = (
    pq.read_table(
        STATE_DATASET_PATH
    )
    .to_pandas()
)

supervised_check = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
)

print(
    "state:",
    state_check.shape
)

print(
    "state signals:",
    state_check[
        "signal_date"
    ].nunique()
)

print(
    "supervised:",
    supervised_check.shape
)

print(
    "supervised signals:",
    supervised_check[
        "signal_date"
    ].nunique()
)

print(
    "target missing:",
    supervised_check[
        "target_return"
    ].isna()
    .sum()
)

state: (21900, 34)
state signals: 438
supervised: (21779, 39)
supervised signals: 436
target missing: 0


05A3
PIT proxy + liquidity + history
→ 매주 투자 후보 50종목

05A4
종목별 asset feature 9개

05C / 05B
market + macro feature 19개

05D
────────────────────────
X = 총 28개 feature

state_dataset
→ 438 × 50
→ RL / inference용
→ 50종목 유지

supervised_dataset
→ target_return 추가
→ Ridge / RF / XGBoost 학습용
→ 관측 불가능 target만 제외